In [1]:
# Imports & Config
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

In [2]:
# Helper Functions
def random_date(start, end):
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

def weighted_choice(values, weights):
    return random.choices(values, weights=weights, k=1)[0]

In [3]:
import random
import pandas as pd
from datetime import datetime, timedelta

first_names = [
    "Parth", "Kaushik", "Kerav", "Prem", "Vaibhav",
    "Amit", "Rohit", "Neha", "Priya", "Ankit",
    "Rahul", "Pooja", "Sneha", "Karan", "Nikita"
]

last_names = [
    "Shah", "Patel", "Mehta", "Verma", "Singh",
    "Joshi", "Desai", "Gupta", "Malhotra", "Iyer"
]

def random_name():
    return f"{random.choice(first_names)} {random.choice(last_names)}"

def generate_email(full_name, user_id):
    name_part = full_name.lower().replace(" ", ".")
    return f"{name_part}{user_id}@example.com"

def random_date(start, end):
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

N_USERS = 1000
cities = ["Mumbai", "Delhi", "Bangalore", "Pune", "Ahmedabad", "Rajkot", "Surat"]

users = []

for i in range(1, N_USERS + 1):
    full_name = random_name()
    
    users.append({
        "user_id": i,
        "full_name": full_name,
        "email": generate_email(full_name, i),
        "city": random.choice(cities),
        "signup_date": random_date(
            datetime(2021, 1, 1),
            datetime(2024, 12, 31)
        )
    })

df_users = pd.DataFrame(users)
df_users.head()


,user_id,full_name,email,city,signup_date
0,1,Rahul Patel,rahul.patel1@example.com,Mumbai,2022-07-18
1,2,Prem Verma,prem.verma2@example.com,Delhi,2021-07-29
2,3,Rahul Malhotra,rahul.malhotra3@example.com,Mumbai,2024-04-24
3,4,Rohit Shah,rohit.shah4@example.com,Mumbai,2021-07-11
4,5,Prem Verma,prem.verma5@example.com,Ahmedabad,2024-05-17


In [4]:
categories = [
    {"category_id": 1, "category_name": "Electronics"},
    {"category_id": 2, "category_name": "Fashion"},
    {"category_id": 3, "category_name": "Home & Kitchen"},
    {"category_id": 4, "category_name": "Books"},
    {"category_id": 5, "category_name": "Beauty"},
]

df_categories = pd.DataFrame(categories)
df_categories




,category_id,category_name
0,1,Electronics
1,2,Fashion
2,3,Home & Kitchen
3,4,Books
4,5,Beauty


In [5]:
# Generate PRODUCTS
products = [
    (1, "Smartphone", 1, 25000),
    (2, "Laptop", 1, 60000),
    (3, "Jeans", 2, 2000),
    (4, "T-Shirt", 2, 800),
    (5, "Mixer Grinder", 3, 4500),
    (6, "Cookbook", 4, 1200),
    (7, "Face Cream", 5, 900),
]

df_products = pd.DataFrame(
    products,
    columns=["product_id", "product_name", "category_id", "list_price"]
)

df_products

,product_id,product_name,category_id,list_price
0,1,Smartphone,1,25000
1,2,Laptop,1,60000
2,3,Jeans,2,2000
3,4,T-Shirt,2,800
4,5,Mixer Grinder,3,4500
5,6,Cookbook,4,1200
6,7,Face Cream,5,900


In [6]:
# Generate ORDERS
orders = []
order_id = 1

for _, user in df_users.iterrows():
    n_orders = np.random.choice(
        [1, 2, 3, 5, 10],
        p=[0.55, 0.20, 0.15, 0.07, 0.03]
    )

    for _ in range(n_orders):
        orders.append({
            "order_id": order_id,
            "user_id": user["user_id"],
            "order_date": random_date(
                pd.to_datetime(user["signup_date"]),
                datetime(2025, 1, 1)
            ),
            "order_status": np.random.choice(
                ["DELIVERED", "CANCELLED"],
                p=[0.92, 0.08]
            )
        })
        order_id += 1

df_orders = pd.DataFrame(orders)
df_orders.head(), len(df_orders)

(   order_id  user_id order_date order_status
 0         1        1 2024-03-12    CANCELLED
 1         2        2 2023-09-23    DELIVERED
 2         3        2 2023-07-18    DELIVERED
 3         4        3 2024-12-26    DELIVERED
 4         5        4 2023-12-07    DELIVERED,
 2115)

In [7]:
# Generate ORDER ITEMS
product_ids = df_products["product_id"].tolist()
product_weights = [0.35, 0.25, 0.1, 0.1, 0.1, 0.05, 0.05]

order_items = []
order_item_id = 1

for _, order in df_orders.iterrows():
    n_items = np.random.choice([1, 2, 3], p=[0.6, 0.3, 0.1])

    for _ in range(n_items):
        pid = weighted_choice(product_ids, product_weights)
        price = df_products.loc[
            df_products["product_id"] == pid, "list_price"
        ].values[0]

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": order["order_id"],
            "product_id": pid,
            "quantity": np.random.randint(1, 3),
            "item_price" : round(price * np.random.uniform(0.9, 1.0), 2)
        })
        order_item_id += 1

df_order_items = pd.DataFrame(order_items)
df_order_items.head(), len(df_order_items)

(   order_item_id  order_id  product_id  quantity  item_price
 0              1         1           1         2    24778.80
 1              2         1           7         2      832.35
 2              3         2           1         1    22692.86
 3              4         3           1         1    23647.35
 4              5         3           2         1    55292.07,
 3154)

In [8]:
# Generate PAYMENTS
payments = []
payment_id = 1

for _, order in df_orders.iterrows():
    status = np.random.choice(["SUCCESS", "FAILED"], p=[0.96, 0.04])

    payments.append({
        "payment_id": payment_id,
        "order_id": order["order_id"],
        "payment_method": np.random.choice(["UPI", "CARD", "NETBANKING"]),
        "payment_status": status,
        "amount_paid": 0 if status == "FAILED" else 1,
        "payment_date": order["order_date"]
    })
    payment_id += 1

df_payments = pd.DataFrame(payments)
df_payments.head()


,payment_id,order_id,payment_method,payment_status,amount_paid,payment_date
0,1,1,CARD,SUCCESS,1,2024-03-12
1,2,2,CARD,SUCCESS,1,2023-09-23
2,3,3,UPI,SUCCESS,1,2023-07-18
3,4,4,NETBANKING,SUCCESS,1,2024-12-26
4,5,5,CARD,SUCCESS,1,2023-12-07


In [9]:
# Sanity Checks
print("Users:", df_users.shape)
print("Orders:", df_orders.shape)
print("Order Items:", df_order_items.shape)
print("Payments:", df_payments.shape)

# FK integrity
assert df_orders["user_id"].isin(df_users["user_id"]).all()
assert df_order_items["order_id"].isin(df_orders["order_id"]).all()
assert df_payments["order_id"].isin(df_orders["order_id"]).all()


Users: (1000, 5)
Orders: (2115, 4)
Order Items: (3154, 5)
Payments: (2115, 6)


In [14]:
df_users.to_csv("../../Data/users.csv", index=False)
df_products.to_csv("../../Data/products.csv", index=False)
df_orders.to_csv("../../Data/orders.csv", index=False)
df_order_items.to_csv("../../Data/order_items.csv", index=False)
df_payments.to_csv("../../Data/payments.csv", index=False)
df_categories.to_csv("../../Data/categories.csv",index=False)

In [15]:
# Row counts
print("Users:", df_users.shape)
print("Orders:", df_orders.shape)
print("Order Items:", df_order_items.shape)
print("Payments:", df_payments.shape)

Users: (1000, 5)
Orders: (2115, 4)
Order Items: (3154, 5)
Payments: (2115, 6)


In [16]:
# FK integrity checks
assert df_orders["user_id"].isin(df_users["user_id"]).all()
assert df_order_items["order_id"].isin(df_orders["order_id"]).all()
assert df_order_items["product_id"].isin(df_products["product_id"]).all()
assert df_payments["order_id"].isin(df_orders["order_id"]).all()

In [17]:
df_products["category_id"].unique()

array([1, 2, 3, 4, 5])